In [17]:
import numpy as np
import pandas as pd

In [18]:
# Datos crudos de descarga
# Se organiza por tamaño de paquete descargado (en bytes), distancia (en metros) 
# y por cada distancia, el tiempo de descarga (en segundos) y la potencia de la señal recibida (RSSI en dBm).

raw = {
    4088: {
        1:  {'tiempo': [0.747, 0.412, 0.467, 0.618], 'rssi': [-48, -51, -51, -45]},
        5:  {'tiempo': [0.649, 0.402, 0.835, 0.410], 'rssi': [-73, -64, -79, -69]},
        10: {'tiempo': [0.391, 0.387, 0.487, 0.426], 'rssi': [-76, -74, -74, -75]},
    },
    36792: {
        1:  {'tiempo': [6.296, 4.630, 7.831, 6.344], 'rssi': [-45, -48, -45, -47]},
        5:  {'tiempo': [6.035, 6.610, 5.565, 5.080], 'rssi': [-74, -65, -65, -80]},
        10: {'tiempo': [5.489, 5.184, 4.972, 5.124], 'rssi': [-79, -83, -82, -82]},
    },
    108332: {
        1:  {'tiempo': [18.783, 18.819, 16.712, 14.431], 'rssi': [-45, -45, -45, -46]},
        5:  {'tiempo': [12.674, 14.218, 20.095, 15.561], 'rssi': [-69, -67, -63, -66]},
        10: {'tiempo': [16.837, 16.452, 16.940, 17.939], 'rssi': [-72, -73, -81, -73]},
    },
    995428: {
        1:  {'tiempo': [172.098, 143.162, 163.141, 142.281], 'rssi': [-47, -47, -48, -47]},
        5:  {'tiempo': [134.540, 150.966, 138.677, 138.666], 'rssi': [-66, -70, -68, -66]},
        10: {'tiempo': [173.591, 161.520, 149.188, 148.259], 'rssi': [-79, -78, -79, -77]},
    },
}

In [19]:
filas = []
for tamano, datos in raw.items():
    for distancia, valores in datos.items():
        for i, (t, r) in enumerate(zip(valores['tiempo'], valores['rssi']), start=1):
            filas.append({
                'paquete_bytes': tamano,
                'distancia_m': distancia,
                'repeticion': i,
                'tiempo_s': t,
                'rssi_dbm': r,
                'throughput_kbps': (tamano / t) / 1024,
            })
df_raw = pd.DataFrame(filas)

pd.options.display.float_format = lambda x: f'{x:.2f}'.replace('.', ',')

df_raw

,paquete_bytes,distancia_m,repeticion,tiempo_s,rssi_dbm,throughput_kbps
0,4088,1,1,"0,75",-48,"5,34"
1,4088,1,2,"0,41",-51,"9,69"
2,4088,1,3,"0,47",-51,"8,55"
3,4088,1,4,"0,62",-45,"6,46"
4,4088,5,1,"0,65",-73,"6,15"
5,4088,5,2,"0,40",-64,"9,93"
6,4088,5,3,"0,83",-79,"4,78"
7,4088,5,4,"0,41",-69,"9,74"
8,4088,10,1,"0,39",-76,"10,21"
9,4088,10,2,"0,39",-74,"10,32"


In [20]:
# A partir de los datos crudos, genero un array con tamaño de paquete, distancia, tiempo de descarga, potencia de la señal recibida (RSSI) y throughput (bytes/segundo)


def construir_variable(tamano_paquete, datos_por_distancia):
    distancias, tiempos, rssis = [], [], []
    for distancia, valores in datos_por_distancia.items():
        n = len(valores['tiempo'])
        distancias.extend([distancia] * n)
        tiempos.extend(valores['tiempo'])
        rssis.extend(valores['rssi'])

    distancia = np.array(distancias, dtype=float)
    tiempo = np.array(tiempos, dtype=float)
    rssi = np.array(rssis, dtype=float)
    throughput = tamano_paquete / tiempo  # bytes/segundo

    return {
        'tamano_paquete': tamano_paquete,
        'distancia': distancia,
        'tiempo': tiempo,
        'rssi': rssi,
        'throughput': throughput,
    }

paquetes = {tamano: construir_variable(tamano, distancias) for tamano, distancias in raw.items()}

pkt_4088   = paquetes[4088]
pkt_36792  = paquetes[36792]
pkt_108332 = paquetes[108332]
pkt_995428 = paquetes[995428]

pkt_4088

{'tamano_paquete': 4088,
 'distancia': array([ 1.,  1.,  1.,  1.,  5.,  5.,  5.,  5., 10., 10., 10., 10.]),
 'tiempo': array([0.747, 0.412, 0.467, 0.618, 0.649, 0.402, 0.835, 0.41 , 0.391,
        0.387, 0.487, 0.426]),
 'rssi': array([-48., -51., -51., -45., -73., -64., -79., -69., -76., -74., -74.,
        -75.]),
 'throughput': array([ 5472.55689424,  9922.33009709,  8753.74732334,  6614.88673139,
         6298.92141757, 10169.15422886,  4895.80838323,  9970.73170732,
        10455.24296675, 10563.30749354,  8394.25051335,  9596.24413146])}

In [21]:
# Primero, miramos la correlacion entre la potencia de la señal y la distancia

for tamano, d in paquetes.items():
    r = np.corrcoef(d['distancia'], d['rssi'])[0, 1]
    print(f"Paquete {tamano:>7} bytes. r = {r:.3f}")

Paquete    4088 bytes. r = -0.856
Paquete   36792 bytes. r = -0.926
Paquete  108332 bytes. r = -0.936
Paquete  995428 bytes. r = -0.968


In [22]:
# Ahora, la correlacion entre el rssi y el tiempo de descarga, y entre el rssi y el throughput
for tamano, datos in raw.items():
    tiempos, rssis = [], []
    for distancia, valores in datos.items():
        tiempos.extend(valores['tiempo'])
        rssis.extend(valores['rssi'])

    tiempos = np.array(tiempos)
    rssis = np.array(rssis)
    throughput = tamano / tiempos  # bytes/segundo

    r_tiempo = np.corrcoef(tiempos, rssis)[0, 1]
    r_throughput = np.corrcoef(throughput, rssis)[0, 1]

    print(f"Paquete {tamano:>7} bytes. r(RSSI,tiempo) = {r_tiempo:.3f}, r(RSSI,throughput) = {r_throughput:.3f}")

Paquete    4088 bytes. r(RSSI,tiempo) = 0.108, r(RSSI,throughput) = -0.182
Paquete   36792 bytes. r(RSSI,tiempo) = 0.567, r(RSSI,throughput) = -0.520
Paquete  108332 bytes. r(RSSI,tiempo) = 0.193, r(RSSI,throughput) = -0.175
Paquete  995428 bytes. r(RSSI,tiempo) = -0.022, r(RSSI,throughput) = 0.030


In [23]:
# Y entre la distancia y el tiempo de descarga, y entre la distancia y el throughput

for tamano, datos in raw.items():
    distancias, tiempos = [], []
    for distancia, valores in datos.items():
        n = len(valores['tiempo'])
        distancias.extend([distancia] * n)
        tiempos.extend(valores['tiempo'])

    distancias = np.array(distancias, dtype=float)
    tiempos = np.array(tiempos)
    throughput = tamano / tiempos

    r_tiempo = np.corrcoef(distancias, tiempos)[0, 1]
    r_throughput = np.corrcoef(distancias, throughput)[0, 1]

    print(f"Paquete {tamano:>7} bytes. r(dist,tiempo) = {r_tiempo:.3f}, r(dist,throughput) = {r_throughput:.3f}")

Paquete    4088 bytes. r(dist,tiempo) = -0.398, r(dist,throughput) = 0.439
Paquete   36792 bytes. r(dist,tiempo) = -0.515, r(dist,throughput) = 0.475
Paquete  108332 bytes. r(dist,tiempo) = -0.007, r(dist,throughput) = -0.033
Paquete  995428 bytes. r(dist,tiempo) = 0.133, r(dist,throughput) = -0.144


In [24]:
# Correlación global, mezclando los 48 puntos, entre distancia y tiempo, y entre distancia y throughput

distancias_todas, tiempos_todos, throughput_todos = [], [], []

for tamano, datos in raw.items():
    for distancia, valores in datos.items():
        for t in valores['tiempo']:
            distancias_todas.append(distancia)
            tiempos_todos.append(t)
            throughput_todos.append(tamano / t)

distancias_todas = np.array(distancias_todas, dtype=float)
tiempos_todos = np.array(tiempos_todos)
throughput_todos = np.array(throughput_todos)

r_dist_tiempo = np.corrcoef(distancias_todas, tiempos_todos)[0, 1]
r_dist_throughput = np.corrcoef(distancias_todas, throughput_todos)[0, 1]

print(f"r(distancia, tiempo)      = {r_dist_tiempo:.3f}")
print(f"r(distancia, throughput)  = {r_dist_throughput:.3f}")

r(distancia, tiempo)      = 0.005
r(distancia, throughput)  = 0.206


In [25]:
# Ahora, las estadísticas de tiempo de descarga por tamaño de paquete

for tamano, d in paquetes.items():
    tiempo = d['tiempo']
    media = np.mean(tiempo)
    sd = np.std(tiempo, ddof=1)   # ddof=1 -> desviación típica muestral
    cv = sd / media * 100         # en %

    print(f"Paquete {tamano:>7} bytes. media = {media:7.3f} s, SD = {sd:6.3f} s, CV = {cv:5.2f} %")

Paquete    4088 bytes. media =   0.519 s, SD =  0.154 s, CV = 29.71 %
Paquete   36792 bytes. media =   5.763 s, SD =  0.901 s, CV = 15.63 %
Paquete  108332 bytes. media =  16.622 s, SD =  2.148 s, CV = 12.92 %
Paquete  995428 bytes. media = 151.341 s, SD = 13.264 s, CV =  8.76 %


In [26]:
# Estadísticas de tiempo de descarga por paquete y distancia
for tamano, datos in raw.items():
    print(f"\nPaquete {tamano} bytes")
    for distancia, valores in datos.items():
        tiempo = np.array(valores['tiempo'])
        media = np.mean(tiempo)
        sd = np.std(tiempo, ddof=1)
        cv = sd / media * 100

        print(f"  {distancia:>2} m. media = {media:7.3f} s, SD = {sd:6.3f} s, CV = {cv:5.2f} %")


Paquete 4088 bytes
   1 m. media =   0.561 s, SD =  0.152 s, CV = 27.01 %
   5 m. media =   0.574 s, SD =  0.208 s, CV = 36.30 %
  10 m. media =   0.423 s, SD =  0.046 s, CV = 10.95 %

Paquete 36792 bytes
   1 m. media =   6.275 s, SD =  1.308 s, CV = 20.84 %
   5 m. media =   5.822 s, SD =  0.654 s, CV = 11.23 %
  10 m. media =   5.192 s, SD =  0.217 s, CV =  4.18 %

Paquete 108332 bytes
   1 m. media =  17.186 s, SD =  2.084 s, CV = 12.13 %
   5 m. media =  15.637 s, SD =  3.198 s, CV = 20.45 %
  10 m. media =  17.042 s, SD =  0.634 s, CV =  3.72 %

Paquete 995428 bytes
   1 m. media = 155.171 s, SD = 14.837 s, CV =  9.56 %
   5 m. media = 140.712 s, SD =  7.108 s, CV =  5.05 %
  10 m. media = 158.139 s, SD = 11.943 s, CV =  7.55 %


In [28]:
tabla_resultados = df_raw.groupby(['paquete_bytes', 'distancia_m']).agg(
    tiempo_medio_s=('tiempo_s', 'mean'),
    rssi_medio_dbm=('rssi_dbm', 'mean'),
    throughput_medio_KBs=('throughput_kbps', lambda x: x.mean()),
).reset_index()

tabla_resultados.style.format({
    'tiempo_medio_s': '{:.2f}',
    'rssi_medio_dbm': '{:.1f}',
    'throughput_medio_KBs': '{:.2f}',
}).hide(axis='index')

paquete_bytes,distancia_m,tiempo_medio_s,rssi_medio_dbm,throughput_medio_KBs
4088,1,0.56,-48.8,7.51
4088,5,0.57,-71.2,7.65
4088,10,0.42,-74.8,9.52
36792,1,6.28,-46.2,5.93
36792,5,5.82,-71.0,6.23
36792,10,5.19,-81.5,6.93
108332,1,17.19,-45.2,6.23
108332,5,15.64,-66.2,6.96
108332,10,17.04,-74.8,6.21
995428,1,155.17,-47.2,6.31
